# Cohort, data splits and NMR representations


In [ ]:
from pathlib import Path
import os
SHARED_NOTEBOOK=os.environ.get("CHD_SHARED_NOTEBOOK",str(Path.cwd()/"00_shared_config.ipynb"))
if not Path(SHARED_NOTEBOOK).exists(): raise FileNotFoundError(SHARED_NOTEBOOK)
get_ipython().run_line_magic("run",f'"{SHARED_NOTEBOOK}"')


In [ ]:
missing_input_paths=[os.path.join(DATA_DIR,name) for name in file_names if not os.path.exists(os.path.join(DATA_DIR,name))]
if missing_input_paths: raise FileNotFoundError("Required UKB input files missing:\n"+"\n".join(missing_input_paths))
input_file_manifest=[]
merge_audit_rows=[]
merge_population_audit_rows=[]
if len(file_names)!=INPUT_FILE_CONTRACT["required_file_count"] or len(set(file_names))!=len(file_names):
    raise RuntimeError("Seven-file input contract is not satisfied")

# 2. Decide columns and read CSV files

wanted_feature_ids = set()
for ids in feature_groups.values():
    wanted_feature_ids.update(ids)
if use_genetic_pcs:
    wanted_feature_ids.add(genetic_pc_id)

baseline_date_ids = [53]
icd10_code_ids = [41270]
icd10_date_ids = [41280]
icd9_code_ids = [41271]
icd9_date_ids = [41281]
death_cause_ids = [40001, 40002]
death_date_ids = [40000]
self_report_ids = [20002]
diabetes_ids = [2443]
medication_summary_ids = [6153, 6177]
medication_detail_ids = [20003]
genetic_qc_ids = reported_sex_ids + [22001, 22006] + qc_indicator_ids

wanted_exclusion_ids = set(
    baseline_date_ids
    + icd10_code_ids + icd10_date_ids
    + icd9_code_ids + icd9_date_ids
    + death_cause_ids + death_date_ids
    + self_report_ids
    + diabetes_ids
    + medication_summary_ids
    + medication_detail_ids
    + genetic_qc_ids
)
all_wanted_ids = wanted_feature_ids | wanted_exclusion_ids

def read_selected_cols(file_path):
    headers = pd.read_csv(file_path, nrows=0).columns.tolist()
    eid_col = find_eid_col(headers)
    if eid_col is None:
        raise ValueError(f"{file_path} has no Eid column")
    cols_to_use = [eid_col]
    for col in headers:
        if col == eid_col:
            continue
        parsed = parse_ukb_col(col)
        if parsed is None:
            continue
        fid, inst, arr = parsed
        if fid in death_cause_ids + death_date_ids:
            cols_to_use.append(col)
            continue
        if inst != 0:
            continue
        if fid in bp_ids:
            cols_to_use.append(col)
        elif use_genetic_pcs and fid == genetic_pc_id and arr in pc_numbers:
            cols_to_use.append(col)
        elif fid in (
            icd10_code_ids + icd10_date_ids
            + icd9_code_ids + icd9_date_ids
            + self_report_ids
            + medication_summary_ids
            + medication_detail_ids
        ):
            cols_to_use.append(col)
        elif fid in all_wanted_ids and arr == 0:
            cols_to_use.append(col)
    if len(cols_to_use) == 1:
        raise RuntimeError(f"Required input file has no contract fields beyond Eid: {file_path}")
    df = pd.read_csv(file_path, usecols=cols_to_use, low_memory=False)
    df = df.rename(columns={eid_col: "Eid"})
    df["Eid"]=df["Eid"].astype("string").str.strip()
    if df["Eid"].isna().any() or df["Eid"].eq("").any():
        raise RuntimeError(f"Missing Eid in required input: {file_path}")
    duplicate_eid_n=int(df["Eid"].duplicated(keep=False).sum())
    if duplicate_eid_n:
        raise RuntimeError(f"Duplicate Eid rows in required input {file_path}: {duplicate_eid_n}")
    df.attrs["source_header_n"]=len(headers)
    df.attrs["selected_column_n"]=len(cols_to_use)
    pass
    return df

dataframes = []
for file_name in file_names:
    path = os.path.join(DATA_DIR, file_name)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing input file: {path}")
        continue
    df_part = read_selected_cols(path)
    if df_part is not None:
        dataframes.append(df_part)
        input_file_manifest.append({
            "name":file_name,
            "path":path,
            "bytes":int(os.path.getsize(path)),
            "mtime_utc":str(pd.Timestamp(os.path.getmtime(path),unit="s",tz="UTC")),
            "n":int(len(df_part)),
            "eid_unique_n":int(df_part["Eid"].nunique()),
            "eid_duplicate_row_n":int(df_part["Eid"].duplicated(keep=False).sum()),
            "source_header_columns":int(df_part.attrs["source_header_n"]),
            "selected_columns":int(df_part.attrs["selected_column_n"]),
            "sha256":sha256_file(path),
        })

if len(dataframes) != INPUT_FILE_CONTRACT["required_file_count"]:
    raise RuntimeError("Input contract expected %d files, loaded %d" % (INPUT_FILE_CONTRACT["required_file_count"],len(dataframes)))

df_final = dataframes[0]
merge_population_audit_rows.append({
    "Step":1,
    "Input_File":file_names[0],
    "Left_N":int(len(df_final)),
    "Right_N":int(len(df_final)),
    "Inner_Merged_N":int(len(df_final)),
    "Dropped_From_Left_N":0,
    "Dropped_From_Right_N":0,
})
for merge_step,df_part in enumerate(dataframes[1:],start=2):
    overlap_cols = sorted((set(df_final.columns) & set(df_part.columns)) - {"Eid"})
    if overlap_cols:
        overlap_check=pd.merge(df_final[["Eid"]+overlap_cols],df_part[["Eid"]+overlap_cols],on="Eid",how="inner",suffixes=("__left","__right"))
        for duplicate in overlap_cols:
            left=overlap_check[duplicate+"__left"]; right=overlap_check[duplicate+"__right"]
            comparable=left.notna() & right.notna(); asymmetric_missing=left.isna() ^ right.isna(); left_number=pd.to_numeric(left,errors="coerce"); right_number=pd.to_numeric(right,errors="coerce")
            numeric=comparable & left_number.notna() & right_number.notna(); nonnumeric=comparable & ~numeric
            numeric_mismatch=~np.isclose(left_number.loc[numeric],right_number.loc[numeric],rtol=1e-10,atol=1e-12,equal_nan=True)
            string_mismatch=left.loc[nonnumeric].astype(str).str.strip()!=right.loc[nonnumeric].astype(str).str.strip()
            mismatches=int(asymmetric_missing.sum()+numeric_mismatch.sum()+string_mismatch.sum())
            merge_audit_rows.append({"column":duplicate,"overlap_n":int(len(overlap_check)),"both_nonmissing_n":int(comparable.sum()),"asymmetric_missing_n":int(asymmetric_missing.sum()),"mismatch_n":mismatches})
            if mismatches: raise RuntimeError(f"Conflicting duplicate field {duplicate}: {mismatches} values")
        pass
        df_part = df_part.drop(columns=overlap_cols)
    left_n=int(len(df_final)); right_n=int(len(df_part))
    merged=pd.merge(df_final,df_part,on="Eid",how="inner",validate="one_to_one")
    merge_population_audit_rows.append({
        "Step":merge_step,
        "Input_File":file_names[merge_step-1],
        "Left_N":left_n,
        "Right_N":right_n,
        "Inner_Merged_N":int(len(merged)),
        "Dropped_From_Left_N":int(left_n-len(merged)),
        "Dropped_From_Right_N":int(right_n-len(merged)),
    })
    df_final=merged

print("-" * 40)

input_manifest_path=os.path.join(AUDIT_DIR,"input_file_manifest.csv")
atomic_write_csv(pd.DataFrame(input_file_manifest),input_manifest_path,index=False)
atomic_write_csv(pd.DataFrame(merge_audit_rows),os.path.join(AUDIT_DIR,"duplicate_field_audit.csv"),index=False)
atomic_write_csv(pd.DataFrame(merge_population_audit_rows),os.path.join(AUDIT_DIR,"input_merge_population_audit.csv"),index=False)
atomic_write_json({
    **INPUT_FILE_CONTRACT,
    "data_directory":DATA_DIR,
    "manifest_file":input_manifest_path,
    "manifest_file_sha256":sha256_file(input_manifest_path),
    "all_required_files_present":len(input_file_manifest)==INPUT_FILE_CONTRACT["required_file_count"],
},os.path.join(AUDIT_DIR,"input_file_contract.json"))


In [ ]:
# 3. Clean feature columns and build feature_cols

for bp_id in bp_ids:
    pattern = rf"^{bp_id}-0\.\d+$"
    bp_cols = [c for c in df_final.columns if re.match(pattern, str(c).strip().replace('"', ""))]
    if len(bp_cols) == 0:
        pass
        continue
    pass
    for c in bp_cols:
        df_final[c] = pd.to_numeric(df_final[c], errors="coerce")
    df_final[str(bp_id)] = df_final[bp_cols].mean(axis=1)
    df_final = df_final.drop(columns=bp_cols)

rename_dict = {}
for col in df_final.columns:
    if col == "Eid":
        continue
    if col in [str(x) for x in bp_ids]:
        continue
    parsed = parse_ukb_col(col)
    if parsed is None:
        continue
    fid, inst, arr = parsed
    if fid == genetic_pc_id and inst == 0 and arr in pc_numbers:
        rename_dict[col] = f"{genetic_pc_id}_PC{arr}"
    elif fid in wanted_feature_ids and inst == 0 and arr == 0:
        rename_dict[col] = str(fid)
df_final = df_final.rename(columns=rename_dict)

feature_cols = []
for _, ids in feature_groups.items():
    for fid in ids:
        feature_cols.append(str(fid))
if use_genetic_pcs:
    for pc in pc_numbers:
        feature_cols.append(f"{genetic_pc_id}_PC{pc}")
feature_cols = list(dict.fromkeys(feature_cols))

missing_feature_cols = [c for c in feature_cols if c not in df_final.columns]
if len(missing_feature_cols) > 0:
    print("-" * 40)
    pass
    print(missing_feature_cols)
    raise ValueError("Required predictors are missing. Check input files and field IDs.")

for c in feature_cols:
    df_final[c] = pd.to_numeric(df_final[c], errors="coerce")

special_missing_values = {"20116": [-3, -1]}
for col, bad_values in special_missing_values.items():
    if col in df_final.columns:
        df_final[col] = df_final[col].replace(bad_values, np.nan)

nmr106_cols = [str(x) for x in NMR_106_FIELD_MAP.values()]
nmr106_feature_cols = set(nmr106_cols)
full249_feature_cols = set(full249_cols)
non_nmr_feature_cols = [c for c in feature_cols if c not in full249_feature_cols]

print("-" * 40)


In [ ]:
# 4. Build and apply exclusion masks

exclude_reasons = {}
baseline_date_cols = get_cols_by_fid(df_final, 53, instance=0)
if len(baseline_date_cols) == 0:
    raise ValueError("Baseline assessment date field 53-0.0 is required to date ICD diagnoses.")
baseline_date_col = baseline_date_cols[0]
baseline_date = pd.to_datetime(df_final[baseline_date_col], errors="coerce")

self_report_cols = get_cols_by_fid(df_final, 20002, instance=0)
mask_self_report_chd = any_value_in_cols(df_final, self_report_cols, baseline_chd_self_report_codes)
exclude_reasons["baseline_chd_self_report_20002"] = mask_self_report_chd

has_icd10_event, event_date_icd10 = get_first_event_date_from_icd(df_final, 41270, 41280, target_icd10_prefixes)
has_icd9_event, event_date_icd9 = get_first_event_date_from_icd(df_final, 41271, 41281, target_icd9_prefixes)
event_date_union = pd.concat([event_date_icd10, event_date_icd9], axis=1).min(axis=1)

history_from_self = mask_self_report_chd.copy()
history_icd10_only = history_from_self | (
    event_date_icd10.notna() & baseline_date.notna() & (event_date_icd10 <= baseline_date)
)
future_event_icd10_only = event_date_icd10.notna() & baseline_date.notna() & (event_date_icd10 > baseline_date)
future_event_icd10_only.loc[history_icd10_only] = False
history_union = history_from_self | (
    event_date_union.notna() & baseline_date.notna() & (event_date_union <= baseline_date)
)
future_event_union = event_date_union.notna() & baseline_date.notna() & (event_date_union > baseline_date)
future_event_union.loc[history_union] = False
icd9_only = has_icd9_event & (~has_icd10_event)
icd9_earlier_than_icd10 = event_date_icd9.notna() & event_date_icd10.notna() & (event_date_icd9 < event_date_icd10)
reclassified_by_icd9 = (history_union != history_icd10_only) | (future_event_union != future_event_icd10_only)

print("-" * 40)

exclude_reasons["baseline_chd_icd10_I20_I25"] = (
    event_date_icd10.notna() & baseline_date.notna() & (event_date_icd10 <= baseline_date)
)
exclude_reasons["baseline_chd_icd9_410_414"] = (
    event_date_icd9.notna() & baseline_date.notna() & (event_date_icd9 <= baseline_date)
)

med_summary_cols = get_cols_by_fid(df_final, 6153, instance=0) + get_cols_by_fid(df_final, 6177, instance=0)
exclude_reasons["baseline_lipid_lowering_6153_6177"] = any_value_in_cols(
    df_final, med_summary_cols, lipid_lowering_values_6153_6177
)
med_detail_cols = get_cols_by_fid(df_final, 20003, instance=0)
exclude_reasons["baseline_lipid_lowering_20003"] = any_value_in_cols(
    df_final, med_detail_cols, lipid_lowering_codes_20003
)
diabetes_cols = get_cols_by_fid(df_final, 2443, instance=0)
exclude_reasons["baseline_diabetes_2443"] = any_value_in_cols(
    df_final, diabetes_cols, diabetes_yes_values_2443
)
diabetes_self_report_codes_main = (
    baseline_diabetes_self_report_codes_conservative
    if EXCLUDE_BASELINE_GESTATIONAL_DIABETES_IN_MAIN
    else baseline_diabetes_self_report_codes
)
exclude_reasons["baseline_diabetes_self_report_20002"] = any_value_in_cols(
    df_final, self_report_cols, diabetes_self_report_codes_main
)

if not EXCLUDE_BASELINE_DIABETES:
    exclude_reasons["baseline_diabetes_2443"] = pd.Series(False, index=df_final.index)
    exclude_reasons["baseline_diabetes_self_report_20002"] = pd.Series(False, index=df_final.index)
if not EXCLUDE_BASELINE_LIPID_LOWERING:
    exclude_reasons["baseline_lipid_lowering_6153_6177"] = pd.Series(False, index=df_final.index)
    exclude_reasons["baseline_lipid_lowering_20003"] = pd.Series(False, index=df_final.index)

if APPLY_GENETIC_QC:
    genetic_sex_cols = get_cols_by_fid(df_final, 22001, instance=0)
    if "31" in df_final.columns:
        sex_reported = pd.to_numeric(df_final["31"], errors="coerce")
    else:
        reported_sex_cols_found = get_cols_by_fid(df_final, 31, instance=0)
        sex_reported = pd.to_numeric(df_final[reported_sex_cols_found[0]], errors="coerce") if len(reported_sex_cols_found) > 0 else None
    if sex_reported is not None and len(genetic_sex_cols) > 0:
        sex_genetic = pd.to_numeric(df_final[genetic_sex_cols[0]], errors="coerce")
        mask_sex_mismatch = sex_reported.notna() & sex_genetic.notna() & (sex_reported != sex_genetic)
    else:
        mask_sex_mismatch = pd.Series(False, index=df_final.index)
    exclude_reasons["qc_22001_sex_mismatch"] = mask_sex_mismatch

    ethnic_cols = get_cols_by_fid(df_final, 22006, instance=0)
    if ALLOWED_22006_VALUES and len(ethnic_cols) > 0:
        ethnic_s = pd.to_numeric(df_final[ethnic_cols[0]], errors="coerce")
        mask_ethnic = ethnic_s.notna() & ~ethnic_s.isin(ALLOWED_22006_VALUES)
    else:
        mask_ethnic = pd.Series(False, index=df_final.index)
    exclude_reasons["qc_22006_genetic_ethnic_grouping"] = mask_ethnic

    for fid in qc_indicator_ids:
        exclude_reasons[f"qc_{fid}_indicator"] = indicator_abnormal_mask(df_final, fid)

# Mutually exclusive sequential counts plus overlap audit.
_reason_names=list(exclude_reasons)
_overlap=np.zeros((len(_reason_names),len(_reason_names)),dtype=int)
for i,left_name in enumerate(_reason_names):
    for j,right_name in enumerate(_reason_names): _overlap[i,j]=int((exclude_reasons[left_name].fillna(False)&exclude_reasons[right_name].fillna(False)).sum())
atomic_write_csv(pd.DataFrame(_overlap,index=_reason_names,columns=_reason_names).reset_index(names="Reason"),os.path.join(AUDIT_DIR,"exclusion_overlap_matrix.csv"),index=False)
_remaining=pd.Series(True,index=df_final.index); sequential_exclusion_rows=[{"Step":"Initial merged records","N":int(len(df_final)),"Excluded_at_step":0}]
for _reason in _reason_names:
    _new=_remaining & exclude_reasons[_reason].fillna(False); _remaining &= ~exclude_reasons[_reason].fillna(False)
    sequential_exclusion_rows.append({"Step":"Exclude: "+_reason,"N":int(_remaining.sum()),"Excluded_at_step":int(_new.sum())})
atomic_write_csv(pd.DataFrame(sequential_exclusion_rows),os.path.join(AUDIT_DIR,"sequential_exclusion_counts.csv"),index=False)

exclude_mask = pd.Series(False, index=df_final.index)
print("-" * 40)

for reason, mask in exclude_reasons.items():
    mask = mask.fillna(False)
    exclude_mask = exclude_mask | mask
    print(f"{reason}: {int(mask.sum())}")

before_exclusion_n = len(df_final)
df_final = df_final.loc[~exclude_mask].copy()
after_exclusion_n = len(df_final)
print("-" * 40)


In [ ]:
# 5. Build full-follow-up CHD Event / Time

def get_all_cause_death_date(df):
    """Use the earliest death date across instances of UK Biobank field 40000."""
    death_date_cols_found = get_cols_by_fid_any_instance(df, 40000)
    if len(death_date_cols_found) == 0:
        return pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")
    death_dates = [
        pd.to_datetime(df[column], errors="coerce")
        for column in death_date_cols_found
    ]
    return rowwise_min_datetime(death_dates, index=df.index)

def get_chd_death_date(df, all_cause_death_date):
    """Identify CAD-related death dates using the cause-of-death fields."""
    cause_columns = (
        get_cols_by_fid_any_instance(df, 40001)
        + get_cols_by_fid_any_instance(df, 40002)
    )
    result = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")
    for cause_column in cause_columns:
        parsed = parse_ukb_col(cause_column)
        paired_date_column = f"40000-{parsed[1]}.0" if parsed is not None else None
        if paired_date_column in df.columns:
            this_date = pd.to_datetime(df[paired_date_column], errors="coerce")
        else:
            this_date = all_cause_death_date

        cause = df[cause_column].astype("string").str.upper().str.strip()
        is_chd = cause.str.startswith(tuple(target_icd10_prefixes), na=False)
        update = is_chd & this_date.notna() & (
            result.isna() | (this_date < result)
        )
        result.loc[update] = this_date.loc[update]
    return result

registry_end_date = pd.to_datetime(REGISTRY_END_DATE_STR)
df_outcome = df_final.copy()
df_outcome["Date_Attending"] = baseline_date.loc[df_outcome.index]

all_cause_death_date = get_all_cause_death_date(df_outcome)
chd_death_date = get_chd_death_date(df_outcome, all_cause_death_date)
first_chd_date = rowwise_min_datetime(
    [event_date_union.loc[df_outcome.index], chd_death_date],
    index=df_outcome.index,
)

registry_end = pd.Series(
    registry_end_date,
    index=df_outcome.index,
    dtype="datetime64[ns]",
)
censor_date = rowwise_min_datetime(
    [registry_end, all_cause_death_date],
    index=df_outcome.index,
)

df_outcome["First_CHD_Date"] = first_chd_date
df_outcome["All_Cause_Death_Date"] = all_cause_death_date
df_outcome["CHD_Death_Date"] = chd_death_date
df_outcome["Censor_Date"] = censor_date

event_mask = (
    df_outcome["First_CHD_Date"].notna()
    & df_outcome["Date_Attending"].notna()
    & df_outcome["Censor_Date"].notna()
    & (df_outcome["First_CHD_Date"] > df_outcome["Date_Attending"])
    & (df_outcome["First_CHD_Date"] <= df_outcome["Censor_Date"])
)

df_outcome["Event"] = event_mask.astype(int)
df_outcome["Outcome_Date"] = df_outcome["Censor_Date"]
df_outcome.loc[event_mask, "Outcome_Date"] = df_outcome.loc[
    event_mask, "First_CHD_Date"
]
df_outcome["Time"] = (
    df_outcome["Outcome_Date"] - df_outcome["Date_Attending"]
).dt.days / 365.25

baseline_chd_leak_mask = (
    df_outcome["First_CHD_Date"].notna()
    & df_outcome["Date_Attending"].notna()
    & (df_outcome["First_CHD_Date"] <= df_outcome["Date_Attending"])
)
invalid_time_mask = df_outcome["Time"].isna() | (df_outcome["Time"] <= 0)

before_filter_n = len(df_outcome)
df_outcome = df_outcome.loc[
    ~baseline_chd_leak_mask & ~invalid_time_mask
].copy()

print("-" * 40)

print(df_outcome["Event"].value_counts().sort_index())
print("Minimum, median and maximum follow-up:",
      df_outcome["Time"].min(),
      df_outcome["Time"].median(),
      df_outcome["Time"].max())

_non_chd_death=(df_outcome["Event"].eq(0)&df_outcome["All_Cause_Death_Date"].notna()&(df_outcome["All_Cause_Death_Date"]>df_outcome["Date_Attending"])&(df_outcome["All_Cause_Death_Date"]<=pd.to_datetime(REGISTRY_END_DATE_STR)))
df_outcome["Competing_Event"]=_non_chd_death.astype(int)
df_outcome["Event_Type"]=np.select([df_outcome.Event.eq(1),df_outcome.Competing_Event.eq(1)],["CHD","Non-CHD death"],default="Censored")


In [ ]:
# Build prespecified clinical variables; clinical missingness never defines the main cohort.
smoking=pd.to_numeric(df_outcome["20116"],errors="coerce")
for name,level in [("smoking_previous",1),("smoking_current",2)]:
    values=pd.Series(np.nan,index=df_outcome.index,dtype=float)
    values.loc[smoking.notna()]=(smoking.loc[smoking.notna()]==level).astype(float)
    df_outcome[name]=values
for column in ["smoking_previous","smoking_current"]:
    if column not in CATEGORICAL_FEATURE_COLS:
        CATEGORICAL_FEATURE_COLS.append(column)

clinical_base=["31","21003","21001","4079","4080","30690","30760","30700","30720","smoking_previous","smoking_current"]
pc_cols=[f"{genetic_pc_id}_PC{index}" for index in pc_numbers]
prs_col=PRS_COL
required_contract_columns=clinical_base+pc_cols+[prs_col]+full249_cols
missing_prespecified=[column for column in required_contract_columns if column not in df_outcome.columns]
if missing_prespecified:
    raise KeyError(f"Required analysis/Full249 contract variables missing: {missing_prespecified}")
if len(full249_cols)!=249 or len(nmr106_cols)!=106:
    raise RuntimeError("Full249/Raw106 field-count contract failed")

# Main cohort: valid outcome + PRS present + all 106 Raw106 fields present.
# The >=90% Raw106 coverage cohort is an independent sensitivity pipeline when selected; it never replaces the primary split.
df_candidate=df_outcome.copy()
df_candidate["Raw106_Nonmissing_N"]=df_candidate[nmr106_cols].notna().sum(axis=1)
df_candidate["Raw106_Nonmissing_Rate"]=df_candidate["Raw106_Nonmissing_N"]/len(nmr106_cols)
df_candidate["PRS_Present"]=pd.to_numeric(df_candidate[prs_col],errors="coerce").notna()
df_candidate["Raw106_Complete"]=df_candidate["Raw106_Nonmissing_N"].eq(len(nmr106_cols))
df_candidate["Main_Strict_Included"]=df_candidate["PRS_Present"] & df_candidate["Raw106_Complete"]
df_candidate["Coverage90_Sensitivity_Included"]=(
    df_candidate["PRS_Present"]
    & df_candidate["Raw106_Nonmissing_Rate"].ge(NMR_COVERAGE90_THRESHOLD)
)
if CHD_COHORT_ID=="prs_raw106_complete":
    df_candidate["Analysis_Included"]=df_candidate["Main_Strict_Included"]
else:
    df_candidate["Analysis_Included"]=df_candidate["Coverage90_Sensitivity_Included"]
if not df_candidate.loc[df_candidate["Main_Strict_Included"],nmr106_cols].notna().all(axis=None):
    raise RuntimeError("Strict main-cohort Raw106 completeness assertion failed")
if not df_candidate.loc[df_candidate["Analysis_Included"],prs_col].notna().all():
    raise RuntimeError("Active cohort PRS completeness assertion failed")

coverage90_frame=df_candidate.loc[
    df_candidate["Coverage90_Sensitivity_Included"],
    ["Eid","Event","Time","PRS_Present","Raw106_Nonmissing_N","Raw106_Nonmissing_Rate","Raw106_Complete","Main_Strict_Included"],
].copy().reset_index(drop=True)
if len(coverage90_frame)==0 or coverage90_frame["Event"].nunique()<2:
    raise RuntimeError("Coverage90 sensitivity cohort is empty or has one outcome class")
coverage90_train,coverage90_val,coverage90_test=split_train_val_test(
    coverage90_frame,
    "sensitivity_prs_raw106_coverage90",
)
coverage90_split_map=pd.concat([
    coverage90_train[["Eid"]].assign(Sensitivity_Split="train"),
    coverage90_val[["Eid"]].assign(Sensitivity_Split="val"),
    coverage90_test[["Eid"]].assign(Sensitivity_Split="test"),
],ignore_index=True)
coverage90_frame=coverage90_frame.merge(
    coverage90_split_map,
    on="Eid",
    how="left",
    validate="one_to_one",
)
coverage90_frame["AnalysisRole"]="sensitivity"
coverage90_frame["Sensitivity_Name"]="prs_raw106_coverage90"
coverage90_frame["Defines_Main_Split"]=False
if coverage90_frame["Sensitivity_Split"].isna().any():
    raise RuntimeError("Coverage90 sensitivity split alignment failed")
coverage90_summary_df=pd.DataFrame([{
    "Sensitivity_Name":"prs_raw106_coverage90",
    "AnalysisRole":"sensitivity",
    "Minimum_Raw106_Nonmissing_Rate":float(NMR_COVERAGE90_THRESHOLD),
    "N":int(len(coverage90_frame)),
    "Events":int(coverage90_frame["Event"].sum()),
    "Train_N":int(coverage90_frame["Sensitivity_Split"].eq("train").sum()),
    "Validation_N":int(coverage90_frame["Sensitivity_Split"].eq("val").sum()),
    "Test_N":int(coverage90_frame["Sensitivity_Split"].eq("test").sum()),
    "Main_Strict_Complete_N":int(coverage90_frame["Main_Strict_Included"].sum()),
    "Sensitivity_Only_N":int((~coverage90_frame["Main_Strict_Included"]).sum()),
    "Defines_Main_Split":False,
}])
atomic_write_csv(
    coverage90_frame,
    os.path.join(COHORT_SENSITIVITY_DIR,"coverage90_sensitivity_eids.csv"),
    index=False,
)
atomic_write_csv(
    coverage90_summary_df,
    os.path.join(COHORT_SENSITIVITY_DIR,"coverage90_sensitivity_summary.csv"),
    index=False,
)

df_model=df_candidate.loc[df_candidate["Analysis_Included"]].copy().reset_index(drop=True)
df_model["Cohort_ID"]=CHD_COHORT_ID
df_model["AnalysisRole"]=ANALYSIS_ROLE
if len(df_model)==0 or df_model["Event"].nunique()<2:
    raise RuntimeError("Active cohort is empty or has one outcome class")
if ANALYSIS_ROLE=="primary" and not df_model["Raw106_Complete"].all():
    raise RuntimeError("Default main cohort is not strictly Raw106 complete")

sample_summary_df=pd.DataFrame([
    {"Stage":"Valid incident-outcome cohort","N":int(len(df_candidate)),"Events":int(df_candidate.Event.sum()),"Cohort_ID":"all_valid","AnalysisRole":"eligibility","Defines_Active_Cohort":False},
    {"Stage":"PRS available","N":int(df_candidate.PRS_Present.sum()),"Events":int(df_candidate.loc[df_candidate.PRS_Present,"Event"].sum()),"Cohort_ID":"prs_present","AnalysisRole":"eligibility","Defines_Active_Cohort":False},
    {"Stage":f"PRS + Raw106 coverage >= {NMR_COVERAGE90_THRESHOLD:.0%} sensitivity","N":int(len(coverage90_frame)),"Events":int(coverage90_frame.Event.sum()),"Cohort_ID":"coverage90","AnalysisRole":"sensitivity","Defines_Active_Cohort":CHD_COHORT_ID=="coverage90"},
    {"Stage":"PRS + Raw106 complete","N":int(df_candidate["Main_Strict_Included"].sum()),"Events":int(df_candidate.loc[df_candidate.Main_Strict_Included,"Event"].sum()),"Cohort_ID":"prs_raw106_complete","AnalysisRole":"primary","Defines_Active_Cohort":CHD_COHORT_ID=="prs_raw106_complete"},
    {"Stage":"Active analysis cohort","N":int(len(df_model)),"Events":int(df_model.Event.sum()),"Cohort_ID":CHD_COHORT_ID,"AnalysisRole":ANALYSIS_ROLE,"Defines_Active_Cohort":True},
])
print(sample_summary_df.to_string(index=False),flush=True)

scenario_name=MAIN_SCENARIO_NAME
df_train,df_val,df_test=split_train_val_test(df_model,scenario_name)

# Preserve raw clinical values for descriptive tables before train-fitted imputation.
_descriptive_columns=["Eid","Event","Time"]+clinical_base+[prs_col]
analysis_clinical_raw=pd.concat([
    df_train[_descriptive_columns].assign(Split="Train"),
    df_val[_descriptive_columns].assign(Split="Validation"),
    df_test[_descriptive_columns].assign(Split="Test"),
],ignore_index=True)

# Fit clinical/PC imputation on train only; validation and test are transform-only.
clinical_imputation_cols=list(dict.fromkeys(clinical_base+pc_cols))
clinical_categorical_cols=[c for c in clinical_imputation_cols if c in {"31","smoking_previous","smoking_current"}]
clinical_continuous_cols=[c for c in clinical_imputation_cols if c not in clinical_categorical_cols]
clinical_preprocessing_rows=[]

for columns,strategy,variable_type in [
    (clinical_continuous_cols,"median","continuous"),
    (clinical_categorical_cols,"most_frequent","categorical"),
]:
    if not columns:
        continue
    raw_parts={
        "train":df_train[columns].replace([np.inf,-np.inf],np.nan).astype(float),
        "val":df_val[columns].replace([np.inf,-np.inf],np.nan).astype(float),
        "test":df_test[columns].replace([np.inf,-np.inf],np.nan).astype(float),
    }
    all_missing=[column for column in columns if raw_parts["train"][column].notna().sum()==0]
    if all_missing:
        raise RuntimeError(f"Clinical columns entirely missing in train: {all_missing}")
    imputer=SimpleImputer(strategy=strategy)
    transformed={
        "train":imputer.fit_transform(raw_parts["train"]),
    }
    transformed["val"]=imputer.transform(raw_parts["val"])
    transformed["test"]=imputer.transform(raw_parts["test"])
    for split_name,frame in [("train",df_train),("val",df_val),("test",df_test)]:
        frame.loc[:,columns]=transformed[split_name]
    for index,column in enumerate(columns):
        clinical_preprocessing_rows.append({
            "Variable":column,
            "Variable_Type":variable_type,
            "Imputation_Strategy":strategy,
            "Train_Fitted_Value":float(imputer.statistics_[index]),
            "Train_Missing_N":int(raw_parts["train"][column].isna().sum()),
            "Train_Missing_Rate":float(raw_parts["train"][column].isna().mean()),
            "Validation_Missing_N":int(raw_parts["val"][column].isna().sum()),
            "Validation_Missing_Rate":float(raw_parts["val"][column].isna().mean()),
            "Test_Missing_N":int(raw_parts["test"][column].isna().sum()),
            "Test_Missing_Rate":float(raw_parts["test"][column].isna().mean()),
        })

if any(frame[clinical_imputation_cols].isna().any().any() for frame in [df_train,df_val,df_test]):
    raise RuntimeError("Clinical train-fitted imputation left missing values")
clinical_preprocessing_df=pd.DataFrame(clinical_preprocessing_rows)
clinical_preprocessing_path=os.path.join(PREPROCESSING_ARTIFACT_DIR,"clinical_train_fitted_imputation.csv")
atomic_write_csv(clinical_preprocessing_df,clinical_preprocessing_path,index=False)
clinical_preprocessing_metadata={
    "contract":"clinical_train_fitted_imputation",
    "fit_split":"train",
    "validation_policy":"transform_only",
    "test_policy":"transform_only",
    "ordered_columns":clinical_imputation_cols,
    "categorical_columns":clinical_categorical_cols,
    "continuous_columns":clinical_continuous_cols,
    "train_eid_sha256":eid_digest(df_train),
    "registry_file":clinical_preprocessing_path,
    "registry_file_sha256":sha256_file(clinical_preprocessing_path),
}
atomic_write_json(
    clinical_preprocessing_metadata,
    os.path.join(PREPROCESSING_ARTIFACT_DIR,"clinical_train_fitted_imputation_metadata.json"),
)


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.util import Surv
from sksurv.metrics import concordance_index_ipcw
from sklearn.exceptions import ConvergenceWarning as SklearnConvergenceWarning
from joblib import Parallel, delayed

NMR_SCORE_N_JOBS = 5

NMR_ALPHA_GRID=np.asarray(NMR_ELASTICNET_ALPHA_GRID,dtype=float)
NMR_L1_RATIO_GRID=np.asarray(NMR_ELASTICNET_L1_RATIOS,dtype=float)
NMR_INNER_FOLDS=int(NMR_ELASTICNET_INNER_CV)
NMR_SELECTION_HORIZON_YEARS=10.0

def survival_y(frame):
    return Surv.from_arrays(
        event=frame["Event"].astype(bool).to_numpy(),
        time=frame["Time"].astype(float).to_numpy(),
    )

def uno_at_horizon(train_reference,eval_frame,risk,horizon=NMR_SELECTION_HORIZON_YEARS):
    tau=min(
        float(horizon),
        float(train_reference.loc[train_reference.Event.eq(0),"Time"].max())-1e-8,
        float(eval_frame.Time.max())-1e-8,
    )
    if tau<=0:
        raise ValueError("Invalid IPCW tau")
    return float(
        concordance_index_ipcw(
            survival_y(train_reference),
            survival_y(eval_frame),
            np.asarray(risk,float),
            tau=tau,
        )[0]
    )

def _fit_one_nmr_l1_ratio(
    l1_index,
    l1_ratio,
    Xi_train,
    Xi_val,
    fold_train,
    fold_val,
    fold,
):
    log_progress(
        f"NMR_score Full249 inner CV fold={fold}/{NMR_INNER_FOLDS} "
        f"l1_ratio={l1_ratio:g}"
    )

    local_scores=np.full(
        len(NMR_ALPHA_GRID),
        np.nan,
        dtype=float,
    )

    with warnings.catch_warnings():
        warnings.simplefilter("error",SklearnConvergenceWarning)

        model=CoxnetSurvivalAnalysis(
            l1_ratio=float(l1_ratio),
            alphas=NMR_ALPHA_GRID,
            max_iter=NMR_ELASTICNET_MAX_ITER,
            tol=1e-8,
        )

        model.fit(
            Xi_train,
            survival_y(fold_train),
        )

    fitted_alpha_count=model.coef_.shape[1]

    if fitted_alpha_count!=len(NMR_ALPHA_GRID):
        log_progress(
            f"Coxnet alpha path truncated: "
            f"requested={len(NMR_ALPHA_GRID)}, "
            f"returned={fitted_alpha_count}"
        )

    for alpha_index in range(fitted_alpha_count):
        beta=np.asarray(
            model.coef_[:,alpha_index],
            dtype=float,
        )

        if np.allclose(beta,0):
            continue

        local_scores[alpha_index]=uno_at_horizon(
            fold_train,
            fold_val,
            Xi_val@beta,
        )

    return l1_index,local_scores

NMR_SCORE_N_JOBS = 5

def fit_nmr_score_nested(train_frame,val_frame,test_frame):
    # NMR_score uses only the ordered Full249 contract. The main cohort itself
    # remains PRS+Raw106 complete; remaining Full249 missingness is handled
    # inside each training fold.
    raw_train=train_frame[full249_cols].replace([np.inf,-np.inf],np.nan).astype(float)
    raw_val=val_frame[full249_cols].replace([np.inf,-np.inf],np.nan).astype(float)
    raw_test=test_frame[full249_cols].replace([np.inf,-np.inf],np.nan).astype(float)
    all_missing=raw_train.columns[raw_train.notna().sum(axis=0).eq(0)].tolist()
    if all_missing:
        raise RuntimeError(f"Full249 fields entirely missing in train: {all_missing}")

    event=train_frame.Event.astype(int).to_numpy()
    cv=StratifiedKFold(
        n_splits=NMR_INNER_FOLDS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )
    scores=np.full(
        (len(NMR_L1_RATIO_GRID),len(NMR_ALPHA_GRID),NMR_INNER_FOLDS),
        np.nan,
        dtype=float,
    )

    def _run_inner_fold(fold, inner_train, inner_val):

        log_progress(
            f"NMR_score Full249 inner CV fold={fold}/{NMR_INNER_FOLDS} started"
        )

        fold_imputer=SimpleImputer(strategy="median")
        fold_scaler=StandardScaler()

        Xi_train=fold_scaler.fit_transform(
            fold_imputer.fit_transform(raw_train.iloc[inner_train])
        )
        Xi_val=fold_scaler.transform(
            fold_imputer.transform(raw_train.iloc[inner_val])
        )

        fold_train=train_frame.iloc[inner_train].reset_index(drop=True)
        fold_val=train_frame.iloc[inner_val].reset_index(drop=True)
        yi=survival_y(fold_train)

        local_scores=np.full(
            (len(NMR_L1_RATIO_GRID),len(NMR_ALPHA_GRID)),
            np.nan,
            dtype=float,
        )

        for l1_index,l1_ratio in enumerate(NMR_L1_RATIO_GRID):

            log_progress(
                f"NMR_score Full249 inner CV fold={fold}/{NMR_INNER_FOLDS} "
                f"l1_ratio={l1_ratio:g}"
            )

            with warnings.catch_warnings():
                warnings.simplefilter(
                    "error",
                    SklearnConvergenceWarning
                )

                model=CoxnetSurvivalAnalysis(
                    l1_ratio=float(l1_ratio),
                    alphas=NMR_ALPHA_GRID,
                    max_iter=NMR_ELASTICNET_MAX_ITER,
                    tol=1e-8,
                )

                model.fit(Xi_train,yi)

            fitted_alpha_count=model.coef_.shape[1]

            if fitted_alpha_count != len(NMR_ALPHA_GRID):
                log_progress(
                    f"Coxnet alpha path truncated: "
                    f"requested={len(NMR_ALPHA_GRID)}, "
                    f"returned={fitted_alpha_count}"
                )

            for alpha_index in range(fitted_alpha_count):

                beta=np.asarray(
                    model.coef_[:,alpha_index],
                    dtype=float
                )

                if np.allclose(beta,0):
                    continue

                local_scores[
                    l1_index,
                    alpha_index
                ]=uno_at_horizon(
                    fold_train,
                    fold_val,
                    Xi_val@beta,
                )

        return fold,local_scores

    fold_results=Parallel(
        n_jobs=12,
        backend="loky",
    )(
        delayed(_run_inner_fold)(
            fold,
            inner_train,
            inner_val,
        )
        for fold,(inner_train,inner_val)
        in enumerate(cv.split(raw_train,event),start=1)
    )

    for fold,local_scores in fold_results:
        scores[:,:,fold-1]=local_scores

    mean_score=np.nanmean(scores,axis=2)
    if not np.isfinite(mean_score).any():
        raise RuntimeError("No valid inner-CV Full249 NMR_score candidate")
    best_index=np.unravel_index(np.nanargmax(mean_score),mean_score.shape)
    best_l1=float(NMR_L1_RATIO_GRID[best_index[0]])
    best_alpha=float(NMR_ALPHA_GRID[best_index[1]])

    boundary_audit={
        "best_l1_ratio":best_l1,
        "best_alpha":best_alpha,
        "l1_first_grid_value":float(NMR_L1_RATIO_GRID[0]),
        "l1_last_grid_value":float(NMR_L1_RATIO_GRID[-1]),
        "alpha_first_grid_value":float(NMR_ALPHA_GRID[0]),
        "alpha_last_grid_value":float(NMR_ALPHA_GRID[-1]),
        "l1_at_first_boundary":bool(best_index[0]==0),
        "l1_at_last_boundary":bool(best_index[0]==len(NMR_L1_RATIO_GRID)-1),
        "alpha_at_first_boundary":bool(best_index[1]==0),
        "alpha_at_last_boundary":bool(best_index[1]==len(NMR_ALPHA_GRID)-1),
        "boundary_policy":NMR_BOUNDARY_POLICY,
    }
    boundary_audit["accepted_parameter_endpoint"]=bool(
        boundary_audit["l1_at_last_boundary"]
        and np.isclose(best_l1,1.0)
    )
    boundary_audit["accepted_parameter_endpoint_note"]=(
        "l1_ratio=1.0 is the legal pure-L1 endpoint; it is audited but does not require upward grid extension."
        if boundary_audit["accepted_parameter_endpoint"]
        else None
    )
    boundary_audit["any_boundary_hit"]=bool(
        boundary_audit["l1_at_first_boundary"]
        or boundary_audit["l1_at_last_boundary"]
        or boundary_audit["alpha_at_first_boundary"]
        or boundary_audit["alpha_at_last_boundary"]
    )
    boundary_audit["search_extension_required"]=bool(
        boundary_audit["l1_at_first_boundary"]
        or boundary_audit["alpha_at_first_boundary"]
        or boundary_audit["alpha_at_last_boundary"]
    )
    boundary_audit["fail_trigger_boundaries"]="l1_first|alpha_first|alpha_last"
    boundary_path=os.path.join(AUDIT_DIR,"nmr_score_boundary_audit.csv")
    atomic_write_csv(pd.DataFrame([boundary_audit]),boundary_path,index=False)
    if boundary_audit["accepted_parameter_endpoint"]:
        log_progress(
            "NMR_score boundary audit | accepted_parameter_endpoint=true | "
            "l1_ratio=1.0 (pure L1)"
        )
    if boundary_audit["search_extension_required"]:
        message=(
            "NMR_score optimum is on a search boundary that can truncate selection; "
            "extend the lower l1_ratio or alpha grid and rerun before final analysis. "
            f"l1_ratio={best_l1:g}, alpha={best_alpha:g}"
        )
        if NMR_BOUNDARY_POLICY=="fail":
            raise RuntimeError(message)
        warnings.warn(message)

    # Refit train-only preprocessing and the selected model after inner-CV selection.
    outer_imputer=SimpleImputer(strategy="median")
    outer_scaler=StandardScaler()
    train_imputed=outer_imputer.fit_transform(raw_train)
    X_train=outer_scaler.fit_transform(train_imputed)
    X_val=outer_scaler.transform(outer_imputer.transform(raw_val))
    X_test=outer_scaler.transform(outer_imputer.transform(raw_test))

    final_model=CoxnetSurvivalAnalysis(
        l1_ratio=best_l1,
        alphas=[best_alpha],
        max_iter=NMR_ELASTICNET_MAX_ITER,
        tol=1e-8,
    )
    with warnings.catch_warnings():
        warnings.simplefilter("error",SklearnConvergenceWarning)
        final_model.fit(X_train,survival_y(train_frame))
    beta=np.asarray(final_model.coef_[:,0],float)
    if np.allclose(beta,0):
        raise RuntimeError("Selected Full249 NMR_score has no non-zero coefficient")

    audit_rows=[]
    for li,l1 in enumerate(NMR_L1_RATIO_GRID):
        for ai,alpha in enumerate(NMR_ALPHA_GRID):
            audit_rows.append({
                "l1_ratio":float(l1),
                "alpha":float(alpha),
                "mean_inner_uno_c_10y":float(mean_score[li,ai]) if np.isfinite(mean_score[li,ai]) else np.nan,
                "valid_folds":int(np.isfinite(scores[li,ai]).sum()),
                "selected":bool((li,ai)==best_index),
                "l1_boundary":bool(li in {0,len(NMR_L1_RATIO_GRID)-1}),
                "alpha_boundary":bool(ai in {0,len(NMR_ALPHA_GRID)-1}),
            })
    audit=pd.DataFrame(audit_rows)
    coefficient=pd.DataFrame({
        "Contract_Order":np.arange(1,len(full249_cols)+1),
        "Field_ID":full249_cols,
        "Coefficient":beta,
        "Selected":np.abs(beta)>1e-12,
        "In_Raw106":pd.Series(full249_cols).isin(set(nmr106_cols)).to_numpy(),
    })
    preprocessing=pd.DataFrame({
        "Contract_Order":np.arange(1,len(full249_cols)+1),
        "Field_ID":full249_cols,
        "Train_Median_Imputation":outer_imputer.statistics_,
        "Train_Standardization_Mean":outer_scaler.mean_,
        "Train_Standardization_Scale":outer_scaler.scale_,
        "Train_Missing_N":raw_train.isna().sum().to_numpy(),
        "Train_Missing_Rate":raw_train.isna().mean().to_numpy(),
        "Validation_Missing_N":raw_val.isna().sum().to_numpy(),
        "Validation_Missing_Rate":raw_val.isna().mean().to_numpy(),
        "Test_Missing_N":raw_test.isna().sum().to_numpy(),
        "Test_Missing_Rate":raw_test.isna().mean().to_numpy(),
    })
    selected_n=int((np.abs(beta)>1e-12).sum())
    metadata={
        "score_name":"NMR_score",
        "source":"train-derived elastic-net Cox score",
        "predictor_contract_name":"Full249",
        "predictor_field_count":len(full249_cols),
        "predictor_field_ids_sha256":canonical_json_digest(full249_cols),
        "source_definition":"UKB standard NMR 249-panel contract: 168 absolute measures plus 81 ratios",
        "source_definition_version":"Full249-v1; Category 220/phase3 companion contract verified 2026-08-24",
        "explicitly_excluded_extension_fields":[20280,20281],
        "explicitly_excluded_extension_meanings":[
            "glucose-lactate",
            "spectrometer-corrected alanine",
        ],
        "criterion":"mean inner-fold Uno IPCW C-index at 10 years",
        "inner_cv_preprocessing":"fold-specific median imputation and standardization",
        "final_preprocessing_fit_split":"train",
        "alpha_grid":NMR_ALPHA_GRID.tolist(),
        "l1_ratio_grid":NMR_L1_RATIO_GRID.tolist(),
        "inner_folds":NMR_INNER_FOLDS,
        "best_l1_ratio":best_l1,
        "best_alpha":best_alpha,
        "best_score":float(mean_score[best_index]),
        "selected_n":selected_n,
        "selected_n_fixed":False,
        "selected_n_note":"Selected_n is determined by train-only inner CV; 39 is not imposed and is only a historical legacy-run count.",
        "boundary_audit":boundary_audit,
        "train_eid_sha256":eid_digest(train_frame),
        "validation_used_for_score_selection":False,
        "test_used_for_score_selection":False,
    }
    return (
        X_train@beta,
        X_val@beta,
        X_test@beta,
        audit,
        coefficient,
        preprocessing,
        metadata,
        boundary_audit,
    )

(
    train_score,
    val_score,
    test_score,
    nmr_cv_audit,
    nmr_coefficients,
    full249_score_preprocessing,
    nmr_score_metadata,
    nmr_score_boundary_audit,
)=fit_nmr_score_nested(df_train,df_val,df_test)

for frame,values in [(df_train,train_score),(df_val,val_score),(df_test,test_score)]:
    frame["NMR_score"]=np.asarray(values,float)

atomic_write_csv(
    nmr_cv_audit,
    os.path.join(AUDIT_DIR,"nmr_score_inner_cv_grid.csv"),
    index=False,
)
atomic_write_csv(
    nmr_coefficients,
    os.path.join(SCORE_ARTIFACT_DIR,"nmr_score_full249_coefficients.csv"),
    index=False,
)
atomic_write_csv(
    full249_score_preprocessing,
    os.path.join(PREPROCESSING_ARTIFACT_DIR,"nmr_score_full249_train_preprocessing.csv"),
    index=False,
)
atomic_write_json(
    nmr_score_metadata,
    os.path.join(SCORE_ARTIFACT_DIR,"nmr_score_metadata.json"),
)

full249_field_contract_df=pd.DataFrame({
    "Contract_Order":np.arange(1,len(full249_cols)+1),
    "Field_ID":[int(value) for value in full249_cols],
    "In_Raw106":[value in set(nmr106_cols) for value in full249_cols],
})
full249_contract_path=os.path.join(SCORE_ARTIFACT_DIR,"full249_field_contract.csv")
atomic_write_csv(full249_field_contract_df,full249_contract_path,index=False)
full249_contract_metadata={
    **FULL249_FIELD_CONTRACT,
    "source_definition":"UKB standard NMR 249-panel contract: 168 absolute measures plus 81 ratios",
    "source_definition_version":"Full249-v1; Category 220/phase3 companion contract verified 2026-08-24",
    "explicitly_excluded_extension_fields":[20280,20281],
    "explicitly_excluded_extension_meanings":[
        "glucose-lactate",
        "spectrometer-corrected alanine",
    ],
    "ordered_field_ids_sha256":canonical_json_digest(full249_cols),
    "contract_file":full249_contract_path,
    "contract_file_sha256":sha256_file(full249_contract_path),
}
atomic_write_json(
    full249_contract_metadata,
    os.path.join(SCORE_ARTIFACT_DIR,"full249_field_contract_metadata.json"),
)

score_registry=[
    {
        "Registry_Entry":"Full249 field contract",
        "Status":"active predictor contract for NMR_score derivation",
        "Source":"UKB fields 23400-23648 inclusive",
        "Field_N":249,
        "Selected_N":None,
        "Selection_Count_Fixed":False,
        "Role":"score predictor contract; not a main-cohort completeness rule",
        "Provenance":"168 absolute measures + 81 ratios; extension fields 20280 and 20281 excluded",
    },
    {
        "Registry_Entry":"NMR_score",
        "Status":"active train-derived score",
        "Source":"Full249 train-only nested elastic-net Cox",
        "Field_N":249,
        "Selected_N":int(nmr_score_metadata["selected_n"]),
        "Selection_Count_Fixed":False,
        "Role":"primary compressed NMR representation",
        "Provenance":"fold-specific inner-CV imputation/scaling; validation and test never select score parameters",
    },
    {
        "Registry_Entry":"Historical selected-count 39",
        "Status":"not used as a fixed score",
        "Source":"legacy run only",
        "Field_N":None,
        "Selected_N":39,
        "Selection_Count_Fixed":False,
        "Role":"provenance clarification only",
        "Provenance":"39 was a previous data-derived selected_n, not a prespecified field or coefficient contract",
    },
]
score_registry_df=pd.DataFrame(score_registry)
score_registry_csv=os.path.join(SCORE_ARTIFACT_DIR,"score_registry.csv")
atomic_write_csv(score_registry_df,score_registry_csv,index=False)
atomic_write_json(
    {
        "registry_version":"2.1.0",
        "entries":score_registry,
        "full249_contract_sha256":full249_contract_metadata["ordered_field_ids_sha256"],
        "nmr_score_metadata_sha256":canonical_json_digest(nmr_score_metadata),
    },
    os.path.join(SCORE_ARTIFACT_DIR,"score_registry.json"),
)

# Spearman pruning is trained on the training split only and uses Raw106 only.
def spearman_prune_train_only(train_frame,columns,threshold=SPEARMAN_ABS_THRESHOLD):
    imputer=SimpleImputer(strategy="median")
    imputed=pd.DataFrame(
        imputer.fit_transform(train_frame[columns]),
        columns=columns,
    )
    corr=imputed.corr(method="spearman").abs()
    kept=[]
    rows=[]
    for column in columns:
        conflict=next(
            (prior for prior in kept if corr.loc[column,prior]>=threshold),
            None,
        )
        keep=conflict is None
        if keep:
            kept.append(column)
        rows.append({
            "Field_ID":column,
            "Keep":keep,
            "Correlated_With":conflict or "",
            "Abs_Spearman":float(corr.loc[column,conflict]) if conflict else np.nan,
            "Threshold":threshold,
        })
    if not kept:
        raise RuntimeError("Spearman pruning removed all Raw106 variables")
    return kept,pd.DataFrame(rows)

spearman_nmr_cols,spearman_audit=spearman_prune_train_only(
    df_train,
    nmr106_cols,
)
atomic_write_csv(
    spearman_audit,
    os.path.join(AUDIT_DIR,"spearman_pruning_audit.csv"),
    index=False,
)

# The coverage90 sensitivity mode requires train-fitted Raw106 imputation.
# This occurs only after the Full249 nested-CV score has been selected, so it
# cannot leak full-train imputation values into the score inner folds.
raw106_parts={
    "train":df_train[nmr106_cols].replace([np.inf,-np.inf],np.nan).astype(float),
    "val":df_val[nmr106_cols].replace([np.inf,-np.inf],np.nan).astype(float),
    "test":df_test[nmr106_cols].replace([np.inf,-np.inf],np.nan).astype(float),
}
raw106_preprocessing_rows=[]
if CHD_COHORT_ID=="coverage90":
    raw106_all_missing=raw106_parts["train"].columns[
        raw106_parts["train"].notna().sum(axis=0).eq(0)
    ].tolist()
    if raw106_all_missing:
        raise RuntimeError(f"Coverage90 Raw106 fields entirely missing in train: {raw106_all_missing}")
    raw106_imputer=SimpleImputer(strategy="median")
    raw106_values={"train":raw106_imputer.fit_transform(raw106_parts["train"])}
    raw106_values["val"]=raw106_imputer.transform(raw106_parts["val"])
    raw106_values["test"]=raw106_imputer.transform(raw106_parts["test"])
    for split_name,frame in [("train",df_train),("val",df_val),("test",df_test)]:
        frame.loc[:,nmr106_cols]=raw106_values[split_name]
    raw106_fit_values=np.asarray(raw106_imputer.statistics_,float)
    raw106_imputation_status="applied_train_median"
else:
    if any(part.isna().any().any() for part in raw106_parts.values()):
        raise RuntimeError("Strict main cohort unexpectedly contains Raw106 missingness")
    raw106_fit_values=raw106_parts["train"].median(axis=0).to_numpy(float)
    raw106_imputation_status="not_needed_strict_complete"

for index,column in enumerate(nmr106_cols):
    raw106_preprocessing_rows.append({
        "Field_ID":column,
        "Cohort_ID":CHD_COHORT_ID,
        "AnalysisRole":ANALYSIS_ROLE,
        "Status":raw106_imputation_status,
        "Imputation_Strategy":"train median" if CHD_COHORT_ID=="coverage90" else "none",
        "Train_Fitted_Value":float(raw106_fit_values[index]),
        "Train_Missing_N":int(raw106_parts["train"][column].isna().sum()),
        "Validation_Missing_N":int(raw106_parts["val"][column].isna().sum()),
        "Test_Missing_N":int(raw106_parts["test"][column].isna().sum()),
    })
raw106_preprocessing_df=pd.DataFrame(raw106_preprocessing_rows)
raw106_preprocessing_path=os.path.join(
    PREPROCESSING_ARTIFACT_DIR,
    "raw106_train_fitted_imputation.csv",
)
atomic_write_csv(raw106_preprocessing_df,raw106_preprocessing_path,index=False)
raw106_preprocessing_metadata={
    "cohort_id":CHD_COHORT_ID,
    "analysis_role":ANALYSIS_ROLE,
    "status":raw106_imputation_status,
    "fit_split":"train" if CHD_COHORT_ID=="coverage90" else "not_applicable",
    "validation_policy":"transform_only",
    "test_policy":"transform_only",
    "ordered_columns":nmr106_cols,
    "registry_file":raw106_preprocessing_path,
    "registry_file_sha256":sha256_file(raw106_preprocessing_path),
}
atomic_write_json(
    raw106_preprocessing_metadata,
    os.path.join(PREPROCESSING_ARTIFACT_DIR,"raw106_train_fitted_imputation_metadata.json"),
)
if any(frame[nmr106_cols].isna().any().any() for frame in [df_train,df_val,df_test]):
    raise RuntimeError("Active-cohort Raw106 preprocessing left missing values")

clinical_cols=pc_cols+clinical_base
final_feature_sets={
    "Clinical_only":clinical_cols,
    "Clinical_plus_PRS":clinical_cols+[prs_col],
    "Clinical_plus_NMR_score":clinical_cols+["NMR_score"],
    "Clinical_plus_PRS_plus_NMR_score":clinical_cols+[prs_col,"NMR_score"],
    "Clinical_plus_PRS_plus_SpearmanNMR":clinical_cols+[prs_col]+spearman_nmr_cols,
    "Clinical_plus_PRS_plus_Raw106NMR":clinical_cols+[prs_col]+nmr106_cols,
    "Clinical_plus_PRS_plus_Full249NMR":clinical_cols+[prs_col]+full249_cols,
}
FEATURE_SET_ROLES={
    "Clinical_only":"primary_benchmark",
    "Clinical_plus_PRS":"primary_benchmark",
    "Clinical_plus_NMR_score":"primary_benchmark",
    "Clinical_plus_PRS_plus_NMR_score":"primary_nmr_representation",
    "Clinical_plus_PRS_plus_SpearmanNMR":"primary_nmr_representation",
    "Clinical_plus_PRS_plus_Raw106NMR":"primary_nmr_representation",
    "Clinical_plus_PRS_plus_Full249NMR":"expanded_panel_sensitivity",
}
if len(final_feature_sets)!=7 or len(nmr106_cols)!=106 or len(full249_cols)!=249:
    raise RuntimeError("Locked feature-set contract failed")
if FEATURE_SET_ROLES["Clinical_plus_PRS_plus_Full249NMR"]!="expanded_panel_sensitivity":
    raise RuntimeError("Full249 feature set must remain sensitivity-only")
feature_set_registry_df=pd.DataFrame([
    {
        "Feature_Set":name,
        "Role":FEATURE_SET_ROLES[name],
        "Feature_N":len(columns),
        "Contains_PRS":prs_col in columns,
        "Raw106_N":sum(column in set(nmr106_cols) for column in columns),
        "Full249_N":sum(column in set(full249_cols) for column in columns),
        "Consumed_By_Result4":False if name=="Clinical_plus_PRS_plus_Full249NMR" else np.nan,
    }
    for name,columns in final_feature_sets.items()
])
atomic_write_csv(
    feature_set_registry_df,
    os.path.join(PIPELINE_ARTIFACT_DIR,"feature_set_registry.csv"),
    index=False,
)


In [ ]:
# Persist immutable split artifacts and a content-addressed pipeline manifest.
for split_name,frame in [("train",df_train),("val",df_val),("test",df_test)]:
    atomic_write_csv(frame,split_artifact_path(split_name),index=False)
atomic_write_json(final_feature_sets,FEATURE_SETS_PATH)
atomic_write_csv(
    sample_summary_df,
    os.path.join(PIPELINE_ARTIFACT_DIR,"final_sample_summary.csv"),
    index=False,
)

manifest_splits={}
for split_name,frame in [("train",df_train),("val",df_val),("test",df_test)]:
    path=split_artifact_path(split_name)
    # Hash the persisted outcomes using the same CSV parser as the validator.
    stored = pd.read_csv(path, usecols=["Eid", "Time", "Event"], low_memory=False)
    if len(stored) != len(frame) or eid_digest(stored) != eid_digest(frame):
        raise RuntimeError(f"{split_name}: population changed during CSV serialization")
    if not np.isfinite(stored["Time"]).all() or not np.array_equal(stored["Event"], frame["Event"]):
        raise RuntimeError(f"{split_name}: invalid serialized outcomes")
    if not np.allclose(stored["Time"], frame["Time"], rtol=0, atol=1e-12):
        raise RuntimeError(f"{split_name}: Time changed during CSV serialization")
    manifest_splits[split_name]={
        "n":int(len(frame)),
        "events":int(frame.Event.sum()),
        "file_sha256":sha256_file(path),
        "eid_sha256":eid_digest(frame),
        "outcome_sha256":dataframe_digest(stored,["Eid","Time","Event"]),
        "clinical_missing_after_train_fitted_imputation":int(frame[clinical_imputation_cols].isna().sum().sum()),
        "prs_missing":int(frame[prs_col].isna().sum()),
        "raw106_missing_cells":int(frame[nmr106_cols].isna().sum().sum()),
    }

if any(item["prs_missing"]!=0 or item["raw106_missing_cells"]!=0 for item in manifest_splits.values()):
    raise RuntimeError("Main split violates the PRS+Raw106-complete cohort contract")
if any(item["clinical_missing_after_train_fitted_imputation"]!=0 for item in manifest_splits.values()):
    raise RuntimeError("Clinical train-fitted imputation contract failed")

core_field_labels={
    31:"sex",21003:"age_at_baseline",20116:"smoking_status",21001:"BMI",
    4079:"diastolic_blood_pressure",4080:"systolic_blood_pressure",
    30690:"total_cholesterol",30760:"HDL_cholesterol",30700:"creatinine",30720:"cystatin_c",
    26227:"CAD_PRS",22009:"genetic_principal_components",
    53:"baseline_assessment_date",41270:"ICD10_diagnosis",41280:"ICD10_diagnosis_date",
    41271:"ICD9_diagnosis",41281:"ICD9_diagnosis_date",
    40000:"death_date",40001:"primary_cause_of_death",40002:"secondary_cause_of_death",
    20002:"self_reported_non_cancer_illness",2443:"doctor_diagnosed_diabetes",
    6153:"medication_male",6177:"medication_female",20003:"treatment_medication_code",
    22001:"genetic_sex",22006:"genetic_ethnic_grouping",22019:"sex_chromosome_aneuploidy",22027:"heterozygosity_missingness_outlier",
}
core_roles={
    31:"clinical",21003:"clinical",20116:"clinical",21001:"clinical",4079:"clinical",4080:"clinical",
    30690:"clinical",30760:"clinical",30700:"clinical",30720:"clinical",26227:"PRS",22009:"genetic_adjustment",
    53:"time_origin",41270:"outcome",41280:"outcome",41271:"outcome",41281:"outcome",40000:"outcome_censoring",
    40001:"outcome",40002:"outcome",20002:"exclusion",2443:"exclusion",6153:"exclusion",6177:"exclusion",
    20003:"exclusion",22001:"genetic_QC",22006:"genetic_QC",22019:"genetic_QC",22027:"genetic_QC",
}
field_registry_rows=[
    {
        "Field_ID":int(field_id),
        "Label":label,
        "Roles":core_roles[int(field_id)],
        "Instance_Array_Rule":("instance 0 arrays 1-10" if int(field_id)==22009 else "see extraction contract; baseline instance 0 except death any instance"),
        "Source_Resolution":"seven-file input contract; see input_file_manifest.csv",
    }
    for field_id,label in core_field_labels.items()
]
_nmr_label_by_id={int(value):name for name,value in NMR_106_FIELD_MAP.items()}
for order,field_id in enumerate(FULL249_FIELD_IDS,start=1):
    roles=["Full249_NMR_score_predictor"]
    if field_id in _nmr_label_by_id:
        roles.append("Raw106_main_cohort_and_representation")
    field_registry_rows.append({
        "Field_ID":field_id,
        "Label":_nmr_label_by_id.get(field_id,""),
        "Roles":";".join(roles),
        "Instance_Array_Rule":"instance 0 array 0",
        "Source_Resolution":"seven-file input contract; see input_file_manifest.csv",
        "Full249_Order":order,
    })
field_registry_df=pd.DataFrame(field_registry_rows).sort_values(["Field_ID","Roles"]).reset_index(drop=True)
field_registry_path=os.path.join(AUDIT_DIR,"field_registry.csv")
atomic_write_csv(field_registry_df,field_registry_path,index=False)
atomic_write_json({
    "registry_version":PROJECT_SCHEMA_VERSION,
    "field_n":int(len(field_registry_df)),
    "field_registry_file":field_registry_path,
    "field_registry_file_sha256":sha256_file(field_registry_path),
    "full249_ordered_ids_sha256":canonical_json_digest(full249_cols),
    "explicitly_excluded_extension_fields":[20280,20281],
},os.path.join(AUDIT_DIR,"field_registry_metadata.json"))

field_contract_audit_df=pd.DataFrame([
    {
        "Contract":"PRS",
        "Expected_Field_N":1,
        "Observed_Field_N":int(prs_col in df_model.columns),
        "Completeness_Rule":"required complete in main cohort",
        "Pass":bool(prs_col in df_model.columns and df_model[prs_col].notna().all()),
    },
    {
        "Contract":"Raw106",
        "Expected_Field_N":106,
        "Observed_Field_N":sum(column in df_model.columns for column in nmr106_cols),
        "Completeness_Rule":("all 106 observed before split" if CHD_COHORT_ID=="prs_raw106_complete" else "coverage >=90% before split; train-fitted median imputation"),
        "Pass":bool(
            len(nmr106_cols)==106
            and all(column in df_model.columns for column in nmr106_cols)
            and all(frame[nmr106_cols].notna().all(axis=None) for frame in [df_train,df_val,df_test])
        ),
    },
    {
        "Contract":"Full249",
        "Expected_Field_N":249,
        "Observed_Field_N":sum(column in df_model.columns for column in full249_cols),
        "Completeness_Rule":"not a cohort rule; fold-specific imputation for NMR_score",
        "Pass":bool(len(full249_cols)==249 and all(column in df_model.columns for column in full249_cols)),
    },
    {
        "Contract":"Clinical",
        "Expected_Field_N":len(clinical_imputation_cols),
        "Observed_Field_N":sum(column in df_model.columns for column in clinical_imputation_cols),
        "Completeness_Rule":"missing allowed; train-fitted imputation",
        "Pass":bool(all(column in df_model.columns for column in clinical_imputation_cols)),
    },
])
if not field_contract_audit_df["Pass"].all():
    raise RuntimeError("Field contract audit failed")
atomic_write_csv(
    field_contract_audit_df,
    os.path.join(AUDIT_DIR,"field_contract_audit.csv"),
    index=False,
)

cohort_contract_audit={
    "active_cohort_id":CHD_COHORT_ID,
    "analysis_role":ANALYSIS_ROLE,
    "default_main_contract_name":MAIN_COHORT_CONTRACT_NAME,
    "strict_main_n":int(df_candidate["Main_Strict_Included"].sum()),
    "strict_main_events":int(df_candidate.loc[df_candidate.Main_Strict_Included,"Event"].sum()),
    "active_n":int(len(df_model)),
    "active_events":int(df_model.Event.sum()),
    "prs_complete":bool(df_model[prs_col].notna().all()),
    "raw106_complete_before_imputation_n":int(df_model["Raw106_Complete"].sum()),
    "raw106_complete_before_imputation_rate":float(df_model["Raw106_Complete"].mean()),
    "raw106_complete_before_imputation":bool(df_model["Raw106_Complete"].all()),
    "raw106_complete_after_preprocessing":bool(all(frame[nmr106_cols].notna().all(axis=None) for frame in [df_train,df_val,df_test])),
    "raw106_preprocessing_status":raw106_preprocessing_metadata["status"],
    "raw106_field_n":len(nmr106_cols),
    "clinical_missing_excludes_participants":False,
    "coverage90_analysis_role":"sensitivity",
    "coverage90_sensitivity_n":int(len(coverage90_frame)),
    "coverage90_sensitivity_only_n":int((~coverage90_frame.Main_Strict_Included).sum()),
    "active_eid_sha256":eid_digest(df_model),
}
atomic_write_json(
    cohort_contract_audit,
    os.path.join(AUDIT_DIR,"cohort_contract_audit.json"),
)

pipeline_manifest={
    "schema_version":PROJECT_SCHEMA_VERSION,
    "notebook_version":NOTEBOOK_VERSION,
    "run_id":RUN_ID,
    "scenario":MAIN_SCENARIO_NAME,
    "cohort_id":CHD_COHORT_ID,
    "active_cohort_id":CHD_COHORT_ID,
    "primary_cohort_id":PRIMARY_COHORT_ID,
    "sensitivity_cohorts":SENSITIVITY_COHORT_IDS,
    "analysis_role":ANALYSIS_ROLE,
    "prs_column":prs_col,
    "nmr106_columns":nmr106_cols,
    "strict_main_n":int(df_candidate["Main_Strict_Included"].sum()),
    "active_n":int(len(df_model)),
    "raw106_complete_before_imputation_rate":float(df_model["Raw106_Complete"].mean()),
    "seed":RANDOM_STATE,
    "registry_end_date":REGISTRY_END_DATE_STR,
    "input_directory":DATA_DIR,
    "input_files":input_file_manifest,
    "input_data_contract":{
        **INPUT_FILE_CONTRACT,
        "required_file_n":len(file_names),
        "all_required_files_present":len(input_file_manifest)==len(file_names),
        "file_names":file_names,
        "source_release_identity":"captured by per-file SHA-256 in input_files",
    },
    "cohort_contract":{
        "name":CHD_COHORT_ID,
        "analysis_role":ANALYSIS_ROLE,
        "default_main_contract_name":MAIN_COHORT_CONTRACT_NAME,
        "prs_required_complete":True,
        "raw106_required_field_n":106,
        "raw106_required_complete_before_split":CHD_COHORT_ID=="prs_raw106_complete",
        "raw106_minimum_coverage_before_split":1.0 if CHD_COHORT_ID=="prs_raw106_complete" else float(NMR_COVERAGE90_THRESHOLD),
        "clinical_missingness_excludes_participants":False,
        "coverage90_analysis_role":"sensitivity",
        "coverage90_independent_full_pipeline_run":True,
        "coverage90_is_active":CHD_COHORT_ID=="coverage90",
        "coverage90_must_not_replace_primary_split":True,
        "coverage90_sensitivity_threshold":float(NMR_COVERAGE90_THRESHOLD),
        "coverage90_sensitivity_artifact":os.path.join(
            COHORT_SENSITIVITY_DIR,
            "coverage90_sensitivity_eids.csv",
        ),
        "coverage90_sensitivity_artifact_sha256":sha256_file(os.path.join(
            COHORT_SENSITIVITY_DIR,
            "coverage90_sensitivity_eids.csv",
        )),
    },
    "cohort_switches":{
        "exclude_baseline_diabetes":EXCLUDE_BASELINE_DIABETES,
        "exclude_baseline_lipid_lowering":EXCLUDE_BASELINE_LIPID_LOWERING,
        "exclude_gestational_diabetes_in_main":EXCLUDE_BASELINE_GESTATIONAL_DIABETES_IN_MAIN,
        "apply_genetic_qc":APPLY_GENETIC_QC,
        "allowed_22006_values":ALLOWED_22006_VALUES,
    },
    "field_semantics":{
        "4079":"diastolic_blood_pressure",
        "4080":"systolic_blood_pressure",
    },
    "field_registry":{
        "file":field_registry_path,
        "file_sha256":sha256_file(field_registry_path),
    },
    "full249_contract":full249_contract_metadata,
    "nmr_score":nmr_score_metadata,
    "score_registry":{
        "csv":score_registry_csv,
        "csv_sha256":sha256_file(score_registry_csv),
        "json":os.path.join(SCORE_ARTIFACT_DIR,"score_registry.json"),
    },
    "clinical_preprocessing":clinical_preprocessing_metadata,
    "raw106_preprocessing":raw106_preprocessing_metadata,
    "spearman_threshold":SPEARMAN_ABS_THRESHOLD,
    "feature_sets":final_feature_sets,
    "feature_set_roles":FEATURE_SET_ROLES,
    "result4_forbidden_feature_sets":["Clinical_plus_PRS_plus_Full249NMR"],
    "splits":manifest_splits,
    "split_rule":{
        "train":TRAIN_SIZE,
        "validation":VAL_SIZE,
        "test":TEST_SIZE,
        "stratified_by":"Event",
        "test_role":"locked final evaluation; never tuning",
    },
}
atomic_write_json(pipeline_manifest,PIPELINE_MANIFEST_PATH)
log_progress(
    f"STAGE 01 DONE | cohort_id={CHD_COHORT_ID} | analysis_role={ANALYSIS_ROLE} | "
    "train-fitted clinical/Raw106 preprocessing and Full249 train-derived NMR_score saved"
)

# Result 1 source data and separate figures.
analysis_df=pd.concat([
    df_train.assign(Split="Train"),
    df_val.assign(Split="Validation"),
    df_test.assign(Split="Test"),
],ignore_index=True)
score_by_eid=analysis_df[["Eid","NMR_score"]].copy()
analysis_descriptive_df=analysis_clinical_raw.merge(
    score_by_eid,
    on="Eid",
    how="left",
    validate="one_to_one",
)
if analysis_descriptive_df["NMR_score"].isna().any():
    raise RuntimeError("NMR_score alignment failed for descriptive data")

prs_available_n=int(df_candidate["PRS_Present"].sum())
cohort_flow_rows=sequential_exclusion_rows+[
    {
        "Step":"Valid positive follow-up",
        "N":int(len(df_candidate)),
        "Excluded_at_step":int(after_exclusion_n-len(df_candidate)),
    },
    {
        "Step":"PRS present",
        "N":prs_available_n,
        "Excluded_at_step":int(len(df_candidate)-prs_available_n),
    },
    {
        "Step":"Primary main cohort: PRS + all 106 Raw106 fields complete",
        "N":int(df_candidate["Main_Strict_Included"].sum()),
        "Excluded_at_step":int(prs_available_n-df_candidate["Main_Strict_Included"].sum()),
        "Cohort_ID":"prs_raw106_complete",
        "AnalysisRole":"primary",
    },
]
if CHD_COHORT_ID=="coverage90":
    cohort_flow_rows.append({
        "Step":f"Active sensitivity cohort: PRS + Raw106 coverage >= {NMR_COVERAGE90_THRESHOLD:.0%}",
        "N":int(len(df_model)),
        "Excluded_at_step":int(prs_available_n-len(df_model)),
        "Cohort_ID":"coverage90",
        "AnalysisRole":"sensitivity",
    })
cohort_flow_rows.extend([
    {"Step":"Active train split","N":int(len(df_train)),"Excluded_at_step":np.nan,"Cohort_ID":CHD_COHORT_ID,"AnalysisRole":ANALYSIS_ROLE},
    {"Step":"Active validation split","N":int(len(df_val)),"Excluded_at_step":np.nan,"Cohort_ID":CHD_COHORT_ID,"AnalysisRole":ANALYSIS_ROLE},
    {"Step":"Active locked test split","N":int(len(df_test)),"Excluded_at_step":np.nan,"Cohort_ID":CHD_COHORT_ID,"AnalysisRole":ANALYSIS_ROLE},
])
cohort_flow=pd.DataFrame(cohort_flow_rows)
atomic_write_csv(
    cohort_flow,
    os.path.join(FIGURE_DATA_DIR,"figure1A_cohort_flow.csv"),
    index=False,
)

fig,ax=plt.subplots(figsize=(12,0.78*len(cohort_flow)+1.8))
ax.axis("off")
y_positions=np.linspace(0.93,0.07,len(cohort_flow))
for index,(y,row) in enumerate(zip(y_positions,cohort_flow.to_dict("records"))):
    label=f"{row['Step']}\nN = {int(row['N']):,}" if pd.notna(row.get("N")) else str(row["Step"])
    ax.text(
        0.50,
        y,
        label,
        ha="center",
        va="center",
        fontsize=13,
        bbox=dict(
            boxstyle="round,pad=0.45",
            facecolor=CNS_PALETTE[index%len(CNS_PALETTE)],
            alpha=0.14,
            edgecolor=CNS_PALETTE[index%len(CNS_PALETTE)],
            linewidth=1.6,
        ),
    )
    if index<len(y_positions)-1:
        ax.annotate(
            "",
            xy=(0.5,y_positions[index+1]+0.025),
            xytext=(0.5,y-0.025),
            arrowprops=dict(arrowstyle="->",lw=1.6,color="#4D4D4D"),
        )
save_figure(fig,RESULT1_FIGURE_DIR,"Figure_1A_cohort_flow")
plt.close(fig)

workflow=[
    ("UKB baseline","Clinical + PRS + Full249 field contract"),
    ("Locked active cohort",("PRS present + complete Raw106" if CHD_COHORT_ID=="prs_raw106_complete" else "Sensitivity: PRS + Raw106 coverage >=90%")),
    ("Train only","Clinical imputation; Full249 nested EN NMR_score; Raw106 Spearman pruning"),
    ("Validation","Downstream hyperparameter/model selection only"),
    ("Locked test","Final discrimination, calibration, DCA, inference"),
]
atomic_write_csv(
    pd.DataFrame(workflow,columns=["Stage","Definition"]),
    os.path.join(FIGURE_DATA_DIR,"figure1B_analysis_workflow.csv"),
    index=False,
)
fig,ax=plt.subplots(figsize=(15,4.3))
ax.axis("off")
for index,(title,detail) in enumerate(workflow):
    x=0.06+index*0.225
    ax.text(
        x,
        0.55,
        f"{title}\n{detail}",
        ha="center",
        va="center",
        fontsize=12.5,
        wrap=True,
        bbox=dict(
            boxstyle="round,pad=0.55",
            facecolor=CNS_PALETTE[index],
            alpha=0.14,
            edgecolor=CNS_PALETTE[index],
            linewidth=1.8,
        ),
    )
    if index<len(workflow)-1:
        ax.annotate(
            "",
            xy=(x+0.145,0.55),
            xytext=(x+0.08,0.55),
            arrowprops=dict(arrowstyle="->",lw=2,color="#4D4D4D"),
        )
save_figure(fig,RESULT1_FIGURE_DIR,"Figure_1B_analysis_workflow")
plt.close(fig)

def standardized_mean_difference(frame,group,column):
    x=pd.to_numeric(frame[column],errors="coerce")
    g=frame[group].astype(bool)
    x0=x[~g].dropna()
    x1=x[g].dropna()
    pooled=np.sqrt((x0.var(ddof=1)+x1.var(ddof=1))/2)
    return float((x1.mean()-x0.mean())/pooled) if np.isfinite(pooled) and pooled>0 else np.nan

def summarize_continuous(series):
    x=pd.to_numeric(series,errors="coerce")
    return f"{x.mean():.2f} ({x.std(ddof=1):.2f})"

def summarize_binary(series):
    x=pd.to_numeric(series,errors="coerce")
    return f"{int((x==1).sum()):,} ({100*(x==1).mean():.1f}%)"

labels={
    "31":"Male sex",
    "21003":"Age, years",
    "21001":"BMI",
    "4079":"Diastolic blood pressure",
    "4080":"Systolic blood pressure",
    "30690":"Total cholesterol",
    "30760":"HDL cholesterol",
    "30700":"Creatinine",
    "30720":"Cystatin C",
    prs_col:"Polygenic risk score",
    "NMR_score":"Train-derived Full249 NMR score",
    "smoking_previous":"Previous smoking",
    "smoking_current":"Current smoking",
}
table_rows=[]
for column in clinical_base+[prs_col,"NMR_score"]:
    binary=column in {"31","smoking_previous","smoking_current"}
    fun=summarize_binary if binary else summarize_continuous
    table_rows.append({
        "Variable":labels.get(column,column),
        "Overall":fun(analysis_descriptive_df[column]),
        "No_incident_CHD":fun(analysis_descriptive_df.loc[analysis_descriptive_df.Event.eq(0),column]),
        "Incident_CHD":fun(analysis_descriptive_df.loc[analysis_descriptive_df.Event.eq(1),column]),
        "SMD_event_vs_nonevent":standardized_mean_difference(
            analysis_descriptive_df,
            "Event",
            column,
        ),
        "Missing_N":int(analysis_descriptive_df[column].isna().sum()),
    })
table1=pd.DataFrame(table_rows)
table1 = table1.rename(columns={
    "No_incident_CHD": "No_incident_CAD",
    "Incident_CHD": "Incident_CAD",
    "SMD_event_vs_nonevent": "SMD_CAD_vs_nonCAD",
})
units={
 'Male sex':'n (%)','Age, years':'years','BMI':'kg/m²','Diastolic blood pressure':'mmHg',
 'Systolic blood pressure':'mmHg','Total cholesterol':'mmol/L','HDL cholesterol':'mmol/L',
 'Creatinine':'µmol/L','Cystatin C':'mg/L','Previous smoking':'n (%)','Current smoking':'n (%)',
 'Polygenic risk score':'raw descriptive scale; modelled per training-set SD',
 'Train-derived Full249 NMR score':'raw derived-score scale; modelled per training-set SD'}
table1.insert(1,'Unit_or_scale',table1['Variable'].map(units).fillna(''))
atomic_write_csv(
    table1,
    os.path.join(TABLE_DIR,"Table_1_baseline_characteristics.csv"),
    index=False,
)

# SMDs are supplementary diagnostics, not a substitute for the workflow panel.
inclusion_frame=df_candidate.copy()
inclusion_frame["Analysis_Included"]=inclusion_frame.Analysis_Included.astype(int)
smd_inclusion=pd.DataFrame([
    {
        "Variable":labels.get(column,column),
        "SMD":standardized_mean_difference(
            inclusion_frame,
            "Analysis_Included",
            column,
        ),
    }
    for column in clinical_base+[prs_col]
    if column in inclusion_frame
])
smd_event=pd.DataFrame([
    {
        "Variable":labels.get(column,column),
        "SMD":standardized_mean_difference(
            analysis_descriptive_df,
            "Event",
            column,
        ),
    }
    for column in clinical_base+[prs_col,"NMR_score"]
])

split_rows=[]
for comparator in ["Validation","Test"]:
    sub=analysis_descriptive_df.loc[
        analysis_descriptive_df.Split.isin(["Train",comparator])
    ].copy()
    sub["Comparator"]=sub.Split.eq(comparator).astype(int)
    for column in clinical_base+[prs_col,"NMR_score"]:
        split_rows.append({
            "Comparison":f"{comparator} vs Train",
            "Variable":labels.get(column,column),
            "SMD":standardized_mean_difference(sub,"Comparator",column),
        })
smd_split=pd.DataFrame(split_rows)

def plot_single_smd(frame,figure_name,figure_title):
    required={"Variable","SMD"}
    if required.difference(frame.columns):
        raise KeyError(f"SMD plot is missing columns: {sorted(required.difference(frame.columns))}")
    if frame.empty:
        raise RuntimeError(f"SMD plot has no rows: {figure_name}")
    if frame["Variable"].duplicated().any():
        duplicates=frame.loc[frame["Variable"].duplicated(keep=False),"Variable"].tolist()
        raise RuntimeError(f"SMD figure must contain one bar per variable: {duplicates}")
    ordered=frame.assign(Abs_SMD=frame.SMD.abs()).sort_values("Abs_SMD")
    fig,ax=plt.subplots(figsize=(10,max(5.5,0.48*len(ordered))))
    ax.barh(
        ordered.Variable,
        ordered.SMD,
        color=np.where(ordered.SMD>=0,CNS_PALETTE[0],CNS_PALETTE[1]),
    )
    ax.axvline(0,color="black",lw=1)
    ax.axvline(0.1,color="#777777",ls="--")
    ax.axvline(-0.1,color="#777777",ls="--")
    ax.set_title(figure_title,pad=10)
    ax.set_xlabel("Standardized mean difference")
    ax.set_ylabel("")
    fig.tight_layout()
    save_figure(fig,RESULT1_SUPPLEMENT_DIR,figure_name)
    plt.close(fig)

for name,frame,title in [
    (
        "Supplementary_SMD_included_vs_excluded",
        smd_inclusion,
        "Included vs excluded",
    ),
    (
        "Supplementary_SMD_event_vs_nonevent",
        smd_event,
        "Incident CHD vs no incident CHD",
    ),
]:
    atomic_write_csv(
        frame,
        os.path.join(FIGURE_DATA_DIR,f"{name}.csv"),
        index=False,
    )
    plot_single_smd(frame,name,title)

# Keep one combined source-data file, but render each split comparison separately.
split_source_name="Supplementary_SMD_split_balance"
atomic_write_csv(
    smd_split,
    os.path.join(FIGURE_DATA_DIR,f"{split_source_name}.csv"),
    index=False,
)
split_figure_specs=[
    (
        "Validation vs Train",
        "Supplementary_SMD_split_balance_Validation_vs_Train",
        "Validation vs train",
    ),
    (
        "Test vs Train",
        "Supplementary_SMD_split_balance_Test_vs_Train",
        "Test vs train",
    ),
]
if set(smd_split["Comparison"].dropna().unique())!={item[0] for item in split_figure_specs}:
    raise RuntimeError("Unexpected split-balance comparison labels")
for comparison,figure_name,figure_title in split_figure_specs:
    comparison_frame=(
        smd_split.loc[smd_split["Comparison"].eq(comparison),["Variable","SMD"]]
        .copy()
        .reset_index(drop=True)
    )
    plot_single_smd(comparison_frame,figure_name,figure_title)

atomic_write_json(
    {
        "status":"success",
        "completed_at":str(pd.Timestamp.now()),
        "pipeline_manifest_sha256":sha256_file(PIPELINE_MANIFEST_PATH),
        "cohort_id":CHD_COHORT_ID,
        "analysis_role":ANALYSIS_ROLE,
        "default_main_cohort_contract":MAIN_COHORT_CONTRACT_NAME,
        "nmr_score_predictor_contract":"Full249",
        "nmr_score_selected_n":int(nmr_score_metadata["selected_n"]),
        "nmr_score_selected_n_fixed":False,
        "outputs":[
            "Figure_1A_cohort_flow",
            "Figure_1B_analysis_workflow",
            "Table_1_baseline_characteristics",
        ],
    },
    os.path.join(RESULT1_DIR,"SUCCESS.json"),
)
print("Result 1 complete:",RESULT1_DIR)
